# 14 · Rule-Based Training Recommendation Engine (Version 1)

**Project:** Enterprise HR AI  

> ### ⚠️ PROMINENT DATA INTEGRITY WARNING
> **SYNTHETIC DATA — employee current-skill possession was not present in any source file and has been simulated using a tenure/training-based heuristic for MVP demonstration purposes only. This must NOT be presented to stakeholders as real observed skill data. Real deployment requires an actual skills inventory (HRIS export, LMS completion records, or self-assessment survey).**

---

---
## Step 1 · Architecture & Staged Implementation Roadmap

Per the project design specification (DOCX), the recommendation engine is built in stages:

### Version 1 (Current Implementation)
- **Methodology:** Deterministic, direct rule-based dictionary lookup.
- **Engine Mechanics:** Each of the 33 benchmark skills is mapped to a concrete, curated course title. For each employee, their top missing skills are mapped directly to actionable training interventions.
- **ML Dependency:** **None.** No `sentence-transformers`, vector databases, or complex embeddings are used in this notebook.

### Version 2 (Planned Future Upgrade)
- **Methodology:** Semantic / Embedding-Based Matching.
- **Planned Mechanics:** When an enterprise course catalog (e.g., Coursera, Udemy Business, or internal LMS with 500+ syllabus descriptions) becomes available, a dense embedding model (such as `sentence-transformers/all-MiniLM-L6-v2`) will generate vector representations of each skill gap and compute cosine similarity against all course description embeddings to rank top-k learning resources.
- **Status:** Documented architectural roadmap; deferred to later phase per DOCX specifications.

---
## Step 2 · Load Employee Skill Gaps

In [1]:
import pandas as pd
import numpy as np
import os

PROC = os.path.join('..', 'data', 'processed')
gaps_file = os.path.join(PROC, 'employee_skill_gaps.csv')

# Load employee skill gaps skipping the header comment
df_gaps = pd.read_csv(gaps_file, comment='#')
print(f'Loaded employee gap records: {len(df_gaps):,} employees (102 Managers excluded per Step 15/16)')
print(f'Columns: {list(df_gaps.columns)}')
print('\nSeverity distribution:')
print(df_gaps['severity'].value_counts())

Loaded employee gap records: 1,368 employees (102 Managers excluded per Step 15/16)
Columns: ['EmployeeNumber', 'JobRole', 'missing_skills', 'gap_count', 'total_required', 'gap_percentage', 'severity']

Severity distribution:
severity
LOW       870
MEDIUM    427
HIGH       71
Name: count, dtype: int64


---
## Step 3 · Curated Training Catalog (33 Concrete Course Recommendations)

Mapping all 33 unique benchmark skills to concrete, specific professional development and technical training courses.

In [2]:
RECOMMENDATION_CATALOG = {
    # ── Foundational & Essential Skills ──
    "Speaking": "Executive Presentation & Public Speaking Masterclass (Toastmasters / Internal Workshop)",
    "Reading Comprehension": "Technical & Regulatory Documentation Analysis Workshop",
    "Active Listening": "Empathetic Leadership & Active Listening for Cross-Functional Collaboration",
    "Critical Thinking": "Strategic Problem Solving & Root Cause Decision Analysis Seminar",
    "Active Learning": "Continuous Professional Learning & Rapid Skill Acquisition Frameworks",
    "Monitoring": "Operational Process Auditing & KPI Performance Monitoring Protocols",
    "Science": "Scientific Methodology, Evidence-Based Rigor & Laboratory Standards Training",
    "Writing": "Business & Technical Writing: Structuring Executive Summaries & Proposals",
    
    # ── Cloud & Engineering Tools (AWS / Cloud) ──
    "Amazon Web Services AWS CloudFormation": "AWS Infrastructure as Code: CloudFormation & CDK Automated Deployments",
    "Amazon Elastic Compute Cloud EC2": "AWS Compute Architecture: Scalable EC2 Fleet Management & Auto-Scaling",
    "Amazon Web Services AWS software": "AWS Solutions Architect: Core Cloud Services, IAM & Architecture Design",
    "Amazon DynamoDB": "NoSQL Database Architecture with AWS DynamoDB: Modeling & Scalability",
    "Amazon Redshift": "Cloud Data Warehousing & High-Performance SQL Analytics with Amazon Redshift",
    
    # ── Enterprise & Office Productivity Software ──
    "Microsoft Office software": "Enterprise Microsoft 365 Productivity & Workflow Automation Bootcamp",
    "Microsoft Excel": "Advanced Excel: Dynamic Arrays, Power Query & Business Financial Modeling",
    "Adobe Acrobat": "Adobe Acrobat Pro: Digital Signatures, Forms & Secure Document Workflows",
    "Microsoft Outlook": "Time Management, Calendar Optimization & Executive Email Triage in Outlook",
    "Google Docs": "Google Workspace Collaboration: Document Co-Authoring & Cloud Governance",
    "Apple macOS": "macOS for Enterprise: Advanced Terminal, Security & Productivity Tooling",
    "Microsoft Access": "Relational Database Design & SQL Querying with Microsoft Access",
    
    # ── Data, Analytics & Development Software ──
    "IBM SPSS Statistics": "Advanced Statistical Inference & Predictive Modeling using IBM SPSS",
    "Eclipse IDE": "Java & Multi-Language Software Development with Eclipse IDE & Git Plugins",
    "ESRI ArcGIS software": "Spatial Data Analytics & Geospatial Mapping with ESRI ArcGIS Pro",
    
    # ── Specialized Healthcare & Industry Software ──
    "MEDITECH software": "MEDITECH EHR Clinical Data Management & Laboratory Information Systems Track",
    
    # ── Design & CAD Software ──
    "Bentley MicroStation": "Bentley MicroStation CAD: 2D/3D Infrastructure Drafting & Asset Modeling",
    "Autodesk AutoCAD": "AutoCAD Essentials: Mechanical/Architectural Drafting & Dimensioning Standards",
    "Adobe Creative Cloud software": "Adobe Creative Cloud Bootcamp: Multi-App Visual Design & Asset Management",
    "Adobe Photoshop": "Commercial Image Retouching & Asset Production with Adobe Photoshop",
    "Adobe InDesign": "Corporate Layout Design, Multi-Page Publishing & Pitch Decks with InDesign",
    "Adobe After Effects": "Motion Graphics, Video Storytelling & Product Animation with After Effects",
    "Adobe Illustrator": "Vector Graphics, Infographics & Brand Asset Design in Adobe Illustrator",
    
    # ── Sales & Marketing Platforms ──
    "HubSpot software": "Inbound Sales & CRM Pipeline Optimization with HubSpot Sales Hub",
    "Facebook": "B2B Social Media Marketing, Meta Business Suite & Targeted Outreach Campaigns"
}

print(f'Total catalog courses configured: {len(RECOMMENDATION_CATALOG)}')
assert len(RECOMMENDATION_CATALOG) == 33, 'Catalog must contain exactly 33 unique courses!'

Total catalog courses configured: 33


---
## Step 4 · Generate Recommendations per Employee

For each employee:
- Extract their missing skills list.
- Select the **top 3 missing skills** (or fewer if less than 3 are missing).
- Map each missing skill directly to its concrete course recommendation.
- If an employee has no skill gaps (0 missing), record `'None - No skill gaps identified'`.

In [3]:
rec_rows = []

for _, row in df_gaps.iterrows():
    emp_id = row['EmployeeNumber']
    role = row['JobRole']
    sev = row['severity']
    missing_raw = row['missing_skills']
    
    if pd.isna(missing_raw) or str(missing_raw).strip() in ('', 'None', 'nan'):
        top3_skills_str = 'None'
        top3_recs_str = 'None - No skill gaps identified'
    else:
        skills = [s.strip() for s in str(missing_raw).split(';') if s.strip()]
        top3_skills = skills[:3]
        top3_recs = [RECOMMENDATION_CATALOG.get(s, f'Targeted Training for {s}') for s in top3_skills]
        
        top3_skills_str = '; '.join(top3_skills)
        top3_recs_str = '; '.join(top3_recs)
        
    rec_rows.append({
        'EmployeeNumber': emp_id,
        'JobRole': role,
        'severity': sev,
        'top_3_missing_skills': top3_skills_str,
        'top_3_recommendations': top3_recs_str
    })

df_recs = pd.DataFrame(rec_rows)
print(f'Total employee recommendation profiles generated: {len(df_recs):,}')
assert len(df_recs) == 1368, f'Expected 1,368 non-manager recommendation profiles, got {len(df_recs)}'

print('\nSample Employee Recommendations:')
for _, r in df_recs.head(5).iterrows():
    print(f'\nEmployee #{r["EmployeeNumber"]} ({r["JobRole"]} - [{r["severity"]} Severity])')
    print(f'  Top Missing Skills : {r["top_3_missing_skills"]}')
    print(f'  Recommendations    : {r["top_3_recommendations"]}')

Total employee recommendation profiles generated: 1,368

Sample Employee Recommendations:

Employee #1 (Sales Executive - [MEDIUM Severity])
  Top Missing Skills : Speaking; Reading Comprehension; Bentley MicroStation
  Recommendations    : Executive Presentation & Public Speaking Masterclass (Toastmasters / Internal Workshop); Technical & Regulatory Documentation Analysis Workshop; Bentley MicroStation CAD: 2D/3D Infrastructure Drafting & Asset Modeling

Employee #2 (Research Scientist - [LOW Severity])
  Top Missing Skills : Reading Comprehension
  Recommendations    : Technical & Regulatory Documentation Analysis Workshop

Employee #4 (Laboratory Technician - [MEDIUM Severity])
  Top Missing Skills : Reading Comprehension; Science; Google Docs
  Recommendations    : Technical & Regulatory Documentation Analysis Workshop; Scientific Methodology, Evidence-Based Rigor & Laboratory Standards Training; Google Workspace Collaboration: Document Co-Authoring & Cloud Governance

Employee #5 

---
## Step 5 · Save Recommendations Dataset (`employee_recommendations.csv`)

Exporting recommendations to `data/processed/employee_recommendations.csv`.  
The synthetic data warning is retained in line 1.

In [4]:
out_file = os.path.join(PROC, 'employee_recommendations.csv')

warning_comment = (
    '# SYNTHETIC DATA — employee current-skill possession was not present in any source file '
    'and has been simulated using a tenure/training-based heuristic for MVP demonstration purposes only. '
    'This must NOT be presented to stakeholders as real observed skill data. Real deployment requires '
    'an actual skills inventory (HRIS export, LMS completion records, or self-assessment survey).\n'
)

with open(out_file, 'w', encoding='utf-8') as f:
    f.write(warning_comment)
    df_recs.to_csv(f, index=False)

file_size = os.path.getsize(out_file)
print(f'Saved recommendations to: {out_file}')
print(f'File size: {file_size:,} bytes')
print(f'Total records: {len(df_recs):,}')

# Round-trip reload verification
df_reload = pd.read_csv(out_file, comment='#')
assert len(df_reload) == 1368, 'Row count mismatch on reload!'
assert list(df_reload.columns) == ['EmployeeNumber', 'JobRole', 'severity', 'top_3_missing_skills', 'top_3_recommendations']
print('CONFIRMED: Round-trip verification passed cleanly with comment handling.')

Saved recommendations to: ..\data\processed\employee_recommendations.csv
File size: 295,968 bytes
Total records: 1,368
CONFIRMED: Round-trip verification passed cleanly with comment handling.
